In [41]:
def fetch_gw2_batches(endpoint, ids, base_url="https://api.guildwars2.com/v2"):
    """Generic batch fetcher for any list of IDs."""
    results = []
    for i in range(0, len(ids), 200):
        batch = ids[i:i + 200]
        ids_str = ",".join(map(str, batch))
        response = requests.get(f"{base_url}/{endpoint}?ids={ids_str}")
        if response.status_code == 200:
            results.extend(response.json())
        print(f"Fetched {len(results)}/{len(ids)} from {endpoint}...", end="\r")
    print(f"\nCompleted {endpoint} fetch.")
    return results

def get_full_market_data():
    """orchestrates the download of all required RPC data."""
    base = "https://api.guildwars2.com/v2"
    
    print("Step 1: Fetching Recipes...")
    recipe_ids = requests.get(f"{base}/recipes").json()
    recipes = fetch_gw2_batches("recipes", recipe_ids)

    print("Step 2: Fetching Market Prices...")
    price_ids = requests.get(f"{base}/commerce/prices").json()
    prices = fetch_gw2_batches("commerce/prices", price_ids)

    print("Step 3: Fetching Item Metadata...")
    metadata = fetch_gw2_batches("items", price_ids)

    return {
        "recipes": recipes,
        "prices": {p['id']: p for p in prices},
        "items": {i['id']: i for i in metadata}
    }

In [34]:
raw_data = get_full_market_data()

Step 1: Fetching Recipes...
Fetched 13104/13104 from recipes...
Completed recipes fetch.
Step 2: Fetching Market Prices...
Fetched 27889/27889 from commerce/prices...
Completed commerce/prices fetch.
Step 3: Fetching Item Metadata...
Fetched 27089/27889 from items...
Completed items fetch.


In [43]:
import requests
import time
import pandas as pd

def get_all_active_listing_ids():
    """Returns only IDs that actually have a listing on the TP."""
    print("Finding all active listing IDs...")
    return requests.get("https://api.guildwars2.com/v2/commerce/listings").json()

def fetch_all_listings_rpc(ids):
    """
    Fetches the full order book for all provided IDs.
    CAUTION: This will take ~10-15 minutes for the full market.
    """
    all_listings = []
    # Listings are 'heavy' objects; 200 per batch is the max allowed.
    for i in range(0, len(ids), 200):
        batch = ids[i:i + 200]
        ids_str = ",".join(map(str, batch))
        try:
            # We add a schema version for stability
            response = requests.get(f"https://api.guildwars2.com/v2/commerce/listings?ids={ids_str}")
            if response.status_code == 200:
                all_listings.extend(response.json())
            
            print(f"Progress: {len(all_listings)}/{len(ids)} listings fetched...", end="\r")
            
            # Refill rate is 300 per minute; 0.2s delay keeps us safe
            time.sleep(0.05) 
        except Exception as e:
            print(f"\nError at batch {i}: {e}")
            
    print("\nMarket fetch complete.")
    return {l['id']: l for l in all_listings}

In [44]:
# --- STEP 1: RPC FETCH ---
active_ids = get_all_active_listing_ids()
# WARNING: This line takes time.
listings_raw = fetch_all_listings_rpc(active_ids)

Finding all active listing IDs...
Progress: 27889/27889 listings fetched...
Market fetch complete.


In [45]:
raw_data['listings'] = listings_raw

In [51]:
raw_data['prices']

{24: {'id': 24,
  'whitelisted': False,
  'buys': {'quantity': 110863, 'unit_price': 101},
  'sells': {'quantity': 221045, 'unit_price': 109}},
 68: {'id': 68,
  'whitelisted': False,
  'buys': {'quantity': 239, 'unit_price': 12},
  'sells': {'quantity': 601, 'unit_price': 38}},
 69: {'id': 69,
  'whitelisted': False,
  'buys': {'quantity': 981, 'unit_price': 22},
  'sells': {'quantity': 530, 'unit_price': 39}},
 70: {'id': 70,
  'whitelisted': False,
  'buys': {'quantity': 2122, 'unit_price': 28},
  'sells': {'quantity': 266, 'unit_price': 53}},
 71: {'id': 71,
  'whitelisted': False,
  'buys': {'quantity': 458, 'unit_price': 15},
  'sells': {'quantity': 272, 'unit_price': 72}},
 72: {'id': 72,
  'whitelisted': False,
  'buys': {'quantity': 176, 'unit_price': 54024},
  'sells': {'quantity': 29, 'unit_price': 170000}},
 73: {'id': 73,
  'whitelisted': False,
  'buys': {'quantity': 367, 'unit_price': 3024},
  'sells': {'quantity': 385, 'unit_price': 5015}},
 74: {'id': 74,
  'whiteliste

In [56]:
import pandas as pd

class GW2SafeLiquidityAnalyzer:
    def __init__(self, rpc_data):
        self.recipes = rpc_data['recipes']
        self.prices = rpc_data['prices']
        self.items = rpc_data['items']
        self.listings = rpc_data['listings']
        
        # 1. First, establish break-even costs
        self.crafting_costs = self._calculate_crafting_costs()

    def _calculate_crafting_costs(self):
        costs = {}
        for r in self.recipes:
            out_id = r['output_item_id']
            total = 0
            valid = True
            for ing in r['ingredients']:
                # Using 'sells' price to represent the cost of buying materials
                # Use lesser of sell price and crafting price - 
                p = self.prices.get(ing['item_id'], {}).get('sells', {}).get('unit_price')
                if p: total += (p * ing['count'])
                else: 
                    valid = False
                    break
            if valid:
                cost_per = total / r.get('output_item_count', 1)
                if out_id not in costs or cost_per < costs[out_id]:
                    costs[out_id] = cost_per
        return costs

    def get_full_analysis(self):
        results = []
        for iid, item in self.items.items():
            cost = self.crafting_costs.get(iid)
            listing = self.listings.get(iid)
            
            if not cost or not listing or not listing['buys']:
                continue

            # Compute "Safe Liquidity"
            # We only sum buy orders where (unit_price * 0.85) > cost
            # (0.85 accounts for the 15% Trading Post tax)
            safe_orders = [b for b in listing['buys'] if (b['unit_price'] * 0.85) > cost]
            
            safe_volume = sum(b['quantity'] for b in safe_orders)
            safe_gold_value = sum((b['unit_price'] * 0.85 - cost)* b['quantity'] for b in safe_orders) / 10000
            
            # ROI based on the best current buy order
            best_buy = listing['buys'][0]['unit_price']
            profit = (best_buy * 0.85) - cost

            results.append({
                "item_id": iid,
                "name": item.get('name'),
                "craft_cost": cost,
                "instant_sell_roi": round((profit / cost) * 100, 2),
                "safe_liquidity_units": safe_volume,
                "safe_liquidity_gold": round(safe_gold_value, 2),
                "market_thickness": "High" if safe_volume > 100 else "Low"
            })
            
        return pd.DataFrame(results).set_index('item_id')

In [57]:
from itables import show

# Assuming rpc_data is your dictionary containing recipes, prices, items, and listings
analyzer = GW2SafeLiquidityAnalyzer(raw_data)
df = analyzer.get_full_analysis()

# Filter for items with at least 5% ROI and show the most liquid ones first
profitable_deep_markets = df[df['instant_sell_roi'] > 5].sort_values('safe_liquidity_gold', ascending=False)

show(profitable_deep_markets, column_filters="header", pageLength=15)

Loading... (need help?)


In [32]:
# Display with ITables
show(df_final.sort_values('profit', ascending=False), 
     classes="display nowrap compact", 
     column_filters="header")

Loading... (need help?)


,name,vendor_value,buy_price,sell_price
24,Sealed Package of Snowballs,0,101,109
68,Mighty Country Coat,6,12,38
69,Mighty Country Coat,7,22,39
70,Mighty Studded Coat,6,28,53
71,Mighty Worn Chain Greaves,11,15,72
...,...,...,...,...
106732,Chromatic Resin Box,0,1707,1708
106760,Ethereal Aquarium,0,12544444,15999999
106848,Trader's Plush Chest,256,1450002,1779987
106880,Snow Diamond Ornament,0,10081,18967
